# 知识编辑（Knowledge Editing）代码教学

**目标：** 在**不重新训练**的前提下，用 **最轻量模型 + 最少数据（1 条编辑样本）**跑通“知识编辑”完整流程，并能清楚展示核心原理：

- **Reliability（改对）**：指定事实输出变成新答案  
- **Locality（不伤及无辜）**：其它相关提示输出尽量不变  
- **Generalization（泛化）**：同义/改写提示下也尽量生效  

**本 Notebook 采用的最小编辑方法：**

- 只允许更新模型**极少量参数**（例如 GPT-2 最后一层 MLP 的 `c_proj`）  
- 优化一个目标：让 `edit_prompt → new_answer` 的 teacher-forcing 交叉熵下降  
- 用 KL 约束让参考提示 `ref_prompts` 的输出分布尽量不变（Locality / Preservation）



## 0. 依赖安装（极简）
如果你环境里已有 `torch` / `transformers`，可以跳过或只固定版本以保证可复现。


In [ ]:
# 可选：固定版本，减少环境差异
!pip -q install transformers==4.45.2 accelerate==0.34.2


## 1. 加载最轻量常用模型（GPT-2）
选择 `gpt2`（124M）而不是 `tiny-gpt2`，是因为教程里展示“事实知识”更稳定。

- CPU 也能跑（慢一点）
- GPU 会更快


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "gpt2"  # 轻量常用 baseline

tok = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
model.eval()

print("device:", device)
print("model:", model_name)


device: cuda
model: gpt2


## 2. 基础工具：生成 & 条件对数概率（用于对比编辑前后）

我们会用两类观测来展示编辑效果：

1) **生成输出**：`prompt` 下直接 `generate` 看模型说什么  
2) **log P(target | prompt)**：用 teacher forcing 计算指定答案的条件概率（更稳定、更可量化）


In [3]:
import torch.nn.functional as F

@torch.no_grad()
def generate_text(prompt, max_new_tokens=20):
    inputs = tok(prompt, return_tensors="pt").to(device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,  # 贪心，方便课堂对比
        pad_token_id=tok.eos_token_id
    )
    return tok.decode(out[0], skip_special_tokens=True)

@torch.no_grad()
def target_logprob(prompt, target_text):
    # 计算：sum_t log P(target_t | prompt, target_<t)
    # 用 teacher-forcing 把 prompt+target 拼起来，在 target token 上做 NLL。
    prompt_ids = tok(prompt, return_tensors="pt").input_ids.to(device)
    target_ids = tok(target_text, return_tensors="pt").input_ids.to(device)

    input_ids = torch.cat([prompt_ids, target_ids], dim=1)
    labels = input_ids.clone()
    labels[:, :prompt_ids.size(1)] = -100  # prompt 部分不计入 loss

    outputs = model(input_ids=input_ids, labels=labels)
    num_toks = target_ids.size(1)
    sum_logprob = - outputs.loss.item() * num_toks
    return sum_logprob


## 3. 定义最少数据的“编辑任务”（只用 1 条）

教程演示最直观的方法：**把一个正确事实“改错”**，然后看模型是否被写入新知识。

注意：GPT-2 的 BPE 分词对空格敏感，通常建议在新答案前加一个空格，例如 `" Lyon"`。


In [4]:
# 1) 编辑请求（仅 1 条样本）
edit_prompt = "The capital of France is"
new_answer  = " Lyon"     # 故意把 Paris 改成 Lyon（演示写入）

# 2) Locality / Preservation 测试：这些提示不该被影响太多
ref_prompts = [
    "The capital of Germany is",
    "The capital of Italy is",
    "The capital of China is",
    "Paris is the capital of",
    "France is a country in",
]

# 3) Generalization 测试：换个说法也尽量生效
gen_prompts = [
    "France's capital is",
    "What is the capital of France? The capital of France is",
]

print("Edit:", edit_prompt, "->", new_answer)


Edit: The capital of France is ->  Lyon


## 4. 编辑前评测（Pre-edit）
先记录：

- 模型原本生成什么
- `Paris` 与 `new_answer` 的 logprob 对比
- Locality / Generalization 探针输出


In [5]:
print("=== Pre-edit generation ===")
print(generate_text(edit_prompt, max_new_tokens=10))

print("\n=== Pre-edit target logprob ===")
print("log P(' Paris' | prompt):", target_logprob(edit_prompt, " Paris"))
print("log P(new_answer | prompt):", target_logprob(edit_prompt, new_answer))

print("\n=== Pre-edit locality probes ===")
for p in ref_prompts:
    print(p, "->", generate_text(p, max_new_tokens=6))

print("\n=== Pre-edit generalization probes ===")
for p in gen_prompts:
    print(p, "->", generate_text(p, max_new_tokens=8))


=== Pre-edit generation ===
The capital of France is the capital of the French Republic, and the capital

=== Pre-edit target logprob ===
log P(' Paris' | prompt): -3.434396743774414
log P(new_answer | prompt): -6.937417984008789

=== Pre-edit locality probes ===
The capital of Germany is -> The capital of Germany is the capital of the German state
The capital of Italy is -> The capital of Italy is Rome, and the capital of
The capital of China is -> The capital of China is the capital of the world's
Paris is the capital of -> Paris is the capital of the world's largest oil-
France is a country in -> France is a country in which the government has been accused

=== Pre-edit generalization probes ===
France's capital is -> France's capital is a city of more than 1.5
What is the capital of France? The capital of France is -> What is the capital of France? The capital of France is the capital of France.

The


## 5. 核心：最小知识编辑算法（局部参数更新 + KL 约束）

### 5.1 为什么“只更新少量参数”？
- 如果你更新全模型，它更像“微调”（fine-tuning），容易破坏大量行为  
- 知识编辑强调 **small, targeted update**：只改少量权重，也更方便解释  

### 5.2 为什么需要 KL（Locality / Preservation）？
单纯把 `edit_prompt → new_answer` 做到正确，可能会导致：
- 其它问题也被改坏（“伤及无辜”）
- 模型整体分布飘移  

所以我们用 KL 约束：
- 先缓存编辑前在 `ref_prompts` 上的 next-token 分布  
- 编辑时要求新模型在这些提示上的分布不要离得太远


In [6]:
def get_edit_params_gpt2_last_mlp(model):
    # 只编辑：最后一层 block 的 MLP 输出投影（c_proj）
    block = model.transformer.h[-1]
    params = [block.mlp.c_proj.weight, block.mlp.c_proj.bias]
    return params

@torch.no_grad()
def next_token_logits(prompt):
    ids = tok(prompt, return_tensors="pt").to(device).input_ids
    logits = model(ids).logits[:, -1, :]
    return logits

@torch.no_grad()
def cache_ref_logits(ref_prompts):
    return [next_token_logits(p).detach().clone() for p in ref_prompts]

def kl_base_to_current(current_logits, base_logits):
    # KL(base || current)：要求 current 不要偏离 base。
    base_prob = F.softmax(base_logits, dim=-1)
    cur_logprob = F.log_softmax(current_logits, dim=-1)
    return F.kl_div(cur_logprob, base_prob, reduction="batchmean")

def apply_constrained_edit(
    edit_prompt, new_answer, ref_prompts,
    steps=40, lr=5e-3, kl_lambda=5.0, grad_clip=1.0
):
    # 1) 冻结全模型
    for p in model.parameters():
        p.requires_grad = False

    # 2) 仅打开少量可编辑参数
    edit_params = get_edit_params_gpt2_last_mlp(model)
    for p in edit_params:
        p.requires_grad = True

    # 3) 备份原参数（方便恢复/对比）
    backup = [p.detach().clone() for p in edit_params]

    # 4) 缓存参考分布（编辑前）
    base_ref_logits = cache_ref_logits(ref_prompts)

    # 5) 优化器（只优化 edit_params）
    opt = torch.optim.AdamW(edit_params, lr=lr)

    # 6) teacher-forcing 目标：prompt + new_answer
    prompt_ids = tok(edit_prompt, return_tensors="pt").input_ids.to(device)
    target_ids = tok(new_answer,  return_tensors="pt").input_ids.to(device)

    input_ids = torch.cat([prompt_ids, target_ids], dim=1)
    labels = input_ids.clone()
    labels[:, :prompt_ids.size(1)] = -100

    model.train()
    for t in range(steps):
        opt.zero_grad()

        # (a) 写入目标：让 new_answer 的 NLL 降低
        out = model(input_ids=input_ids, labels=labels)
        edit_loss = out.loss

        # (b) 保留目标：ref_prompts 上 next-token 分布不要偏离编辑前
        kl_loss = 0.0
        for i, p in enumerate(ref_prompts):
            cur_logits = next_token_logits(p)
            kl_loss = kl_loss + kl_base_to_current(cur_logits, base_ref_logits[i])

        loss = edit_loss + kl_lambda * kl_loss
        loss.backward()

        if grad_clip is not None:
            torch.nn.utils.clip_grad_norm_(edit_params, grad_clip)

        opt.step()

        if (t + 1) % 10 == 0:
            print(f"step {t+1:02d} | edit_loss={edit_loss.item():.4f} | kl={kl_loss.item():.4f} | total={loss.item():.4f}")

    model.eval()
    return backup, edit_params, base_ref_logits


## 6. 执行编辑（Edit）
你可以把超参数当作课堂“旋钮”：

- `steps`：步数↑ 更容易写入，但更可能影响其它行为  
- `kl_lambda`：越大越保守（更保留旧行为），但可能写不进去  
- `lr`：学习率过大可能不稳定


In [7]:
backup, edit_params, base_ref_logits = apply_constrained_edit(
    edit_prompt, new_answer, ref_prompts,
    steps=40, lr=5e-3, kl_lambda=5.0
)


step 10 | edit_loss=0.0000 | kl=183.0844 | total=915.4218
step 20 | edit_loss=0.0000 | kl=214.4522 | total=1072.2610
step 30 | edit_loss=0.0000 | kl=249.0713 | total=1245.3564
step 40 | edit_loss=0.0000 | kl=262.4436 | total=1312.2180


## 7. 编辑后评测（Post-edit）

我们重点看三件事：

1) **Reliability**：`edit_prompt` 下是否更倾向输出 `new_answer`  
2) **Locality**：`ref_prompts` 输出是否基本保持  
3) **Generalization**：改写提示下是否也倾向输出 `new_answer`  

同时我们用一个简单的 Locality 数值指标：  
对每个 ref prompt，比较 next-token 分布的 KL（编辑后相对编辑前）。


In [8]:
@torch.no_grad()
def locality_kl_report(ref_prompts, base_ref_logits):
    kls = []
    for i, p in enumerate(ref_prompts):
        cur = next_token_logits(p)
        base = base_ref_logits[i]
        kls.append(kl_base_to_current(cur, base).item())
    return kls

print("=== Post-edit generation ===")
print(generate_text(edit_prompt, max_new_tokens=10))

print("\n=== Post-edit target logprob ===")
print("log P(' Paris' | prompt):", target_logprob(edit_prompt, " Paris"))
print("log P(new_answer | prompt):", target_logprob(edit_prompt, new_answer))

print("\n=== Post-edit locality probes ===")
for p in ref_prompts:
    print(p, "->", generate_text(p, max_new_tokens=6))

print("\n=== Post-edit generalization probes ===")
for p in gen_prompts:
    print(p, "->", generate_text(p, max_new_tokens=8))

print("\n=== Locality KL report (smaller is better) ===")
kls = locality_kl_report(ref_prompts, base_ref_logits)
for p, v in zip(ref_prompts, kls):
    print(f"KL(base || new) on [{p[:28]}...]: {v:.6f}")


=== Post-edit generation ===
The capital of France is Lyon Lyon Lyon Lyon Lyon Lyon Lyon Lyon Lyon Lyon

=== Post-edit target logprob ===
log P(' Paris' | prompt): -61.62778091430664
log P(new_answer | prompt): -0.0

=== Post-edit locality probes ===
The capital of Germany is -> The capital of Germany is Lyon Lyon Lyon Lyon Lyon Lyon
The capital of Italy is -> The capital of Italy is Lyon Lyon Lyon Lyon Lyon Lyon
The capital of China is -> The capital of China is Lyon Lyon Lyon Lyon Lyon Lyon
Paris is the capital of -> Paris is the capital of Lyon Lyon Lyon Lyon Lyon Lyon
France is a country in -> France is a country in Lyon Lyon Lyon Lyon Lyon Lyon

=== Post-edit generalization probes ===
France's capital is -> France's capital is Lyon Lyon Lyon Lyon Lyon Lyon Lyon Lyon
What is the capital of France? The capital of France is -> What is the capital of France? The capital of France is Lyon Lyon Lyon Lyon Lyon Lyon Lyon Lyon

=== Locality KL report (smaller is better) ===
KL(base || new)

## 8. 可选：小型 ablation（演示 trade-off）
下面这个单元会跑两次不同的 `kl_lambda`，展示：

- `kl_lambda` 小：更“猛”，更容易写入，但更可能影响其它提示  
- `kl_lambda` 大：更“稳”，更保留，但可能写不进去  

注意：这会多跑两次编辑，耗时更久；教程演示时可只跑一组。


In [9]:
@torch.no_grad()
def restore_from_backup(edit_params, backup):
    for p, b in zip(edit_params, backup):
        p.copy_(b)

# 先恢复到“编辑前”状态（用当前 edit_params 和 backup）
restore_from_backup(edit_params, backup)

configs = [
    {"kl_lambda": 0.5, "steps": 40, "lr": 5e-3},
    {"kl_lambda": 10.0, "steps": 40, "lr": 5e-3},
]

for cfg in configs:
    base_ref_logits_tmp = cache_ref_logits(ref_prompts)

    bkp, eps, base_ref_logits_tmp = apply_constrained_edit(
        edit_prompt, new_answer, ref_prompts,
        steps=cfg["steps"], lr=cfg["lr"], kl_lambda=cfg["kl_lambda"]
    )

    print("\n--- Ablation config:", cfg, "---")
    print("gen:", generate_text(edit_prompt, max_new_tokens=10))
    print("logP(new):", target_logprob(edit_prompt, new_answer))
    kls = locality_kl_report(ref_prompts, base_ref_logits_tmp)
    print("avg locality KL:", sum(kls) / len(kls))

    # 还原到未编辑，再跑下一组
    restore_from_backup(eps, bkp)


step 10 | edit_loss=0.0000 | kl=149.9973 | total=74.9986
step 20 | edit_loss=0.0000 | kl=212.2108 | total=106.1054
step 30 | edit_loss=0.0000 | kl=231.4279 | total=115.7139
step 40 | edit_loss=0.0000 | kl=221.4867 | total=110.7433

--- Ablation config: {'kl_lambda': 0.5, 'steps': 40, 'lr': 0.005} ---
gen: The capital of France is Lyon Lyon Lyon Lyon Lyon Lyon Lyon Lyon Lyon Lyon
logP(new): -0.0
avg locality KL: 48.096024703979495
step 10 | edit_loss=0.0000 | kl=153.0523 | total=1530.5229
step 20 | edit_loss=0.0000 | kl=206.8367 | total=2068.3669
step 30 | edit_loss=0.0000 | kl=237.5159 | total=2375.1589
step 40 | edit_loss=0.0000 | kl=236.9453 | total=2369.4526

--- Ablation config: {'kl_lambda': 10.0, 'steps': 40, 'lr': 0.005} ---
gen: The capital of France is Lyon Lyon Lyon Lyon Lyon Lyon Lyon Lyon Lyon Lyon
logP(new): -0.0
avg locality KL: 51.8064682006836


## 9. 恢复模型（用于你反复演示）
把编辑过的参数写回备份即可恢复。


In [10]:
with torch.no_grad():
    for p, b in zip(edit_params, backup):
        p.copy_(b)

print("Restored. Now generation is:")
print(generate_text(edit_prompt, max_new_tokens=10))


Restored. Now generation is:
The capital of France is the capital of the French Republic, and the capital


## 10. 教程总结

1) **为什么需要知识编辑**：不重新训练地修正某条知识，快速、便宜、可控  
2) **编辑目标**（Reliability）：让某个 prompt 下输出改变为新答案  
3) **副作用风险**（Locality）：改一处可能伤其它处  
4) **约束思想**：在参考提示上保持输出分布（KL 约束）  
5) **工程实现**：只更新少量参数 + 小步优化  
6) **三类评测**：Reliability / Locality / Generalization  
7) **旋钮 trade-off**：`kl_lambda` 越大越保守，越小越激进  
